In [13]:
# Importing libraries

import xarray as xr
import numpy as np
import glob
import pandas as pd
import dask
import os
import gc
import netCDF4
from pathlib import Path
import re
from pandas import ExcelWriter
import openpyxl
import os
import pyarrow as pa
import pyarrow.parquet as pq


In [2]:
# Managing Memory Usage and File Locking

dask.config.set({
    'array.slicing.split_large_chunks': True,  
    'distributed.worker.memory.target': 0.7,   
    'distributed.worker.memory.spill': 0.9,       
    'distributed.comm.timeouts.connect': '300s',
    'distributed.comm.timeouts.tcp': '300s'
})

os.environ['HDF5_USE_FILE_LOCKING'] = 'FALSE'

In [3]:
# Creating list of all files

path = "C:/Users/idasi/Documents/ideas_tih/IDEAS-TIH-Summer-2026/data/raw"
files = sorted((glob.glob(os.path.join(path, "*.nc"))))
print(f"Total files found: {len(files)}")

Total files found: 240


In [4]:
# Extracting and printing time values from each file and writing time unknown if values are not found

'''
for file in files:
    try:
        ds = xr.open_dataset(file)
        time_values = ds['valid_time'].values
        time = pd.DataFrame({
            'File': [file],
            'Time Values': [time_values]
        })
        print(time)
    except (RuntimeError, OSError, KeyError) as e:
        time = pd.DataFrame({
            'File': [file],
            'Time Values': ["Time unknown"]
        })
        print(time)
'''

'\nfor file in files:\n    try:\n        ds = xr.open_dataset(file)\n        time_values = ds[\'valid_time\'].values\n        time = pd.DataFrame({\n            \'File\': [file],\n            \'Time Values\': [time_values]\n        })\n        print(time)\n    except (RuntimeError, OSError, KeyError) as e:\n        time = pd.DataFrame({\n            \'File\': [file],\n            \'Time Values\': ["Time unknown"]\n        })\n        print(time)\n'

In [5]:
#

dset = xr.open_mfdataset(
    files,
    parallel=True,
    engine='netcdf4',
    data_vars='minimal',
    chunks={'valid_time': 720, 'latitude': 125, 'longitude': 121}
    )

print(dset)

print("Chunks:", dset['t2m'].chunks)
print("Type:", type(dset['t2m'].data))


c:\Users\idasi\.conda\envs\ideastih\Lib\site-packages\dask\_task_spec.py:767: UserWarning: The specified chunks separate the stored chunks along dimension "valid_time" starting at index 720. This could degrade performance. Instead, consider rechunking after loading.
  return self.func(*new_argspec, **kwargs)
c:\Users\idasi\.conda\envs\ideastih\Lib\site-packages\dask\_task_spec.py:767: UserWarning: The specified chunks separate the stored chunks along dimension "valid_time" starting at index 720. This could degrade performance. Instead, consider rechunking after loading.
  return self.func(*new_argspec, **kwargs)
c:\Users\idasi\.conda\envs\ideastih\Lib\site-packages\dask\_task_spec.py:767: UserWarning: The specified chunks separate the stored chunks along dimension "valid_time" starting at index 720. This could degrade performance. Instead, consider rechunking after loading.
  return self.func(*new_argspec, **kwargs)
c:\Users\idasi\.conda\envs\ideastih\Lib\site-packages\dask\_task_spec.

<xarray.Dataset> Size: 37GB
Dimensions:     (valid_time: 87672, latitude: 125, longitude: 121)
Coordinates:
  * valid_time  (valid_time) datetime64[ns] 701kB 2016-01-01 ... 2025-12-31T2...
    expver      (valid_time) <U4 1MB dask.array<chunksize=(720,), meta=np.ndarray>
  * latitude    (latitude) float64 1kB 36.0 35.75 35.5 35.25 ... 5.5 5.25 5.0
  * longitude   (longitude) float64 968B 63.0 63.25 63.5 ... 92.5 92.75 93.0
    number      int64 8B 0
Data variables:
    ssrd        (valid_time, latitude, longitude) float32 5GB dask.array<chunksize=(720, 125, 121), meta=np.ndarray>
    strd        (valid_time, latitude, longitude) float32 5GB dask.array<chunksize=(720, 125, 121), meta=np.ndarray>
    t2m         (valid_time, latitude, longitude) float32 5GB dask.array<chunksize=(720, 125, 121), meta=np.ndarray>
    sst         (valid_time, latitude, longitude) float32 5GB dask.array<chunksize=(720, 125, 121), meta=np.ndarray>
    tcc         (valid_time, latitude, longitude) float32 5GB 

C:\Users\idasi\AppData\Local\Temp\ipykernel_18508\3970400964.py:3: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  dset = xr.open_mfdataset(


In [7]:
# Merging files with different variables but same time coordinates and saving merged files 

"""
data_dir = r'C:\Users\idasi\Documents\ideas_tih\IDEAS-TIH-Summer-2026\data\raw'

for accum in glob.glob(os.path.join(data_dir, '*accum*.nc')):
    inst = accum.replace('accum', 'instant')
    
    if os.path.exists(inst):
        merged = xr.merge([xr.open_dataset(accum), xr.open_dataset(inst)])
        merged.to_netcdf(accum.replace('accum', 'merged'))
        merged.close()
        print(f"Merged {os.path.basename(accum)} + {os.path.basename(inst)}")
"""

SyntaxError: (unicode error) 'unicodeescape' codec can't decode bytes in position 16-17: truncated \UXXXXXXXX escape (1437599817.py, line 3)

In [8]:
proc_path = "C:/Users/idasi/Documents/ideas_tih/IDEAS-TIH-Summer-2026/data/processed"
proc_files = sorted((glob.glob(os.path.join(proc_path, "*.nc"))))
print(f"Total files found: {len(proc_files)}")

Total files found: 120


In [14]:
output_dir = os.path.join(os.getcwd(), 'flat_data_parquet')
os.makedirs(output_dir, exist_ok=True)

variables = ['lsm', 'z', 'ssrd', 'strd', 'tcc', 'sst', 't2m']
time_chunk_size = 720

total_files_processed = 0

for var in variables:
    if var not in dset.data_vars:
        print(f"Skipping '{var}' — not in dataset")
        continue

    filepath = os.path.join(output_dir, f'{var}.parquet')
    darray = dset[var]
    writer = None

    try:
        if 'valid_time' not in darray.dims:
            df = darray.to_dataframe().reset_index()
            table = pa.Table.from_pandas(df, preserve_index=False)
            pq.write_table(table, filepath, compression='snappy')
            print(f"Saved '{var}' -> {filepath} ({len(df):,} rows)")
        else:
            total_times = darray.sizes['valid_time']
            total_rows = 0

            for start in range(0, total_times, time_chunk_size):
                end = min(start + time_chunk_size, total_times)
                chunk_df = darray.isel(valid_time=slice(start, end)).to_dataframe().reset_index()
                table = pa.Table.from_pandas(chunk_df, preserve_index=False)

                if writer is None:
                    writer = pq.ParquetWriter(filepath, table.schema, compression='snappy')

                writer.write_table(table)
                total_rows += len(chunk_df)

            print(f"Saved '{var}' -> {filepath} ({total_rows:,} rows total)")
    finally:
        if writer is not None:
            writer.close()

    total_files_processed += 1
    print(f"Files processed so far: {total_files_processed}/{len(variables)}")

Saved 'lsm' -> c:\Users\idasi\Documents\ideas_tih\IDEAS-TIH-Summer-2026\notebook\flat_data_parquet\lsm.parquet (1,326,039,000 rows total)
Files processed so far: 1/7
Saved 'z' -> c:\Users\idasi\Documents\ideas_tih\IDEAS-TIH-Summer-2026\notebook\flat_data_parquet\z.parquet (1,326,039,000 rows total)
Files processed so far: 2/7
Saved 'ssrd' -> c:\Users\idasi\Documents\ideas_tih\IDEAS-TIH-Summer-2026\notebook\flat_data_parquet\ssrd.parquet (1,326,039,000 rows total)
Files processed so far: 3/7
Saved 'strd' -> c:\Users\idasi\Documents\ideas_tih\IDEAS-TIH-Summer-2026\notebook\flat_data_parquet\strd.parquet (1,326,039,000 rows total)
Files processed so far: 4/7
Saved 'tcc' -> c:\Users\idasi\Documents\ideas_tih\IDEAS-TIH-Summer-2026\notebook\flat_data_parquet\tcc.parquet (1,326,039,000 rows total)
Files processed so far: 5/7
Saved 'sst' -> c:\Users\idasi\Documents\ideas_tih\IDEAS-TIH-Summer-2026\notebook\flat_data_parquet\sst.parquet (1,326,039,000 rows total)
Files processed so far: 6/7
Save

Grid size: 15125, time steps per sheet: 66


Exception ignored in: <function ZipFile.__del__ at 0x000001F722FE6A20>
Traceback (most recent call last):
  File "c:\Users\idasi\.conda\envs\ideastih\Lib\zipfile\__init__.py", line 1988, in __del__
    self.close()
  File "c:\Users\idasi\.conda\envs\ideastih\Lib\zipfile\__init__.py", line 2005, in close
    self.fp.seek(self.start_dir)
ValueError: seek of closed file
